In [1]:
'''
def calcUtility(p, i):
    t = (p.sum() / np.sqrt((p*p).sum()) ) * np.sqrt((250/i))
    u = min(max(t, 0), 6) * p.sum()
    return u


date = dlabels.loc[y.index, 'date'].values
weight = dlabels.loc[y.index, 'weight'].values
resp = dlabels.loc[y.index, 'resp'].values
action = estimator.predict(X)

p_i = np.bincount(date, weight * resp * action)
t = p_i.sum() / np.sqrt((p_i ** 2).sum()) * np.sqrt(250 / p_i.size)
u = np.clip(t, 0, 6) * p_i.sum()

return u * scale
'''

"\ndef calcUtility(p, i):\n    t = (p.sum() / np.sqrt((p*p).sum()) ) * np.sqrt((250/i))\n    u = min(max(t, 0), 6) * p.sum()\n    return u\n\n\ndate = dlabels.loc[y.index, 'date'].values\nweight = dlabels.loc[y.index, 'weight'].values\nresp = dlabels.loc[y.index, 'resp'].values\naction = estimator.predict(X)\n\np_i = np.bincount(date, weight * resp * action)\nt = p_i.sum() / np.sqrt((p_i ** 2).sum()) * np.sqrt(250 / p_i.size)\nu = np.clip(t, 0, 6) * p_i.sum()\n\nreturn u * scale\n"

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier

In [3]:
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df[col] = df[col].astype(np.float16)
                    elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df[col] = df[col].astype(np.float32)
                    else:
                        df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))

    return df

def nulls(df):
    unitrainvals = df.isnull().sum().values.tolist()
    traincols = df.columns.values.tolist()
    nUni = list(zip(unitrainvals, traincols))
    print(*nUni, sep = "\n") 

In [4]:
train_raw = reduce_mem_usage(pd.read_csv('../input/jane-street-market-prediction/train.csv'))

Memory usage after optimization is: 631.49 MB
Decreased by 74.9%


In [5]:
train_raw.shape

(2390491, 138)

In [6]:
import gc
gc.collect()

40

In [7]:
train = train_raw.fillna(0)

train['action'] = ((train['weight'].values * train['resp'].values) > 0).astype('int')

In [8]:
nulls(train)

(0, 'date')
(0, 'weight')
(0, 'resp_1')
(0, 'resp_2')
(0, 'resp_3')
(0, 'resp_4')
(0, 'resp')
(0, 'feature_0')
(0, 'feature_1')
(0, 'feature_2')
(0, 'feature_3')
(0, 'feature_4')
(0, 'feature_5')
(0, 'feature_6')
(0, 'feature_7')
(0, 'feature_8')
(0, 'feature_9')
(0, 'feature_10')
(0, 'feature_11')
(0, 'feature_12')
(0, 'feature_13')
(0, 'feature_14')
(0, 'feature_15')
(0, 'feature_16')
(0, 'feature_17')
(0, 'feature_18')
(0, 'feature_19')
(0, 'feature_20')
(0, 'feature_21')
(0, 'feature_22')
(0, 'feature_23')
(0, 'feature_24')
(0, 'feature_25')
(0, 'feature_26')
(0, 'feature_27')
(0, 'feature_28')
(0, 'feature_29')
(0, 'feature_30')
(0, 'feature_31')
(0, 'feature_32')
(0, 'feature_33')
(0, 'feature_34')
(0, 'feature_35')
(0, 'feature_36')
(0, 'feature_37')
(0, 'feature_38')
(0, 'feature_39')
(0, 'feature_40')
(0, 'feature_41')
(0, 'feature_42')
(0, 'feature_43')
(0, 'feature_44')
(0, 'feature_45')
(0, 'feature_46')
(0, 'feature_47')
(0, 'feature_48')
(0, 'feature_49')
(0, 'feature_50'

In [9]:
drop_cols = ['date', 'resp_1', 'resp_2', 'resp_3', 'resp_4', 'resp', 'ts_id', 'weight']

In [10]:
train.drop(drop_cols, inplace=True, axis=1)

In [11]:
#val = train.iloc[2000000: , :]
#train = train.iloc[:2000000 , :]

#gc.collect()

In [12]:
#val

In [13]:
x = train.iloc[:, :130]
y = train.iloc[:, 130:]

In [14]:
from sklearn import metrics

In [15]:
from sklearn.model_selection  import StratifiedKFold
import time

In [16]:
%%time
err = [] 
#y_pred_tot_lgm = np.zeros((len(test), 2))


fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=2020)
i = 1

for train_index, test_index in fold.split(x, y):
    x_train, x_val = x.iloc[train_index], x.iloc[test_index]
    y_train, y_val = y.iloc[train_index], y.iloc[test_index]
    m = CatBoostClassifier(n_estimators=10000,
                       #random_state=150303,
                       eval_metric='AUC',
                       learning_rate=0.08,
                       #depth=8,
                       #bagging_temperature=0.3,
                       task_type='GPU',
                           loss_function='Logloss',
                           use_best_model=True,
                           random_seed=6006,
                           metric_period=1
                       #num_leaves=64
                       
                       )
    m.fit(x_train, y_train,eval_set=[(x_val, y_val)], early_stopping_rounds=100,verbose=200)
    pred_y = m.predict(x_val)
    print(i, " err_lgm: ", metrics.accuracy_score(y_val,pred_y))
    err.append(metrics.roc_auc_score(y_val,pred_y))
    #y_pred_tot_lgm+= m.predict_proba(test)
    i = i + 1
#y_pred_tot_lgm=y_pred_tot_lgm/10
sum(err)/10

0:	learn: 0.5720079	test: 0.5713070	best: 0.5713070 (0)	total: 51.8ms	remaining: 8m 38s
200:	learn: 0.6259363	test: 0.6242788	best: 0.6242788 (200)	total: 8.23s	remaining: 6m 41s
400:	learn: 0.6363834	test: 0.6325937	best: 0.6325937 (400)	total: 16s	remaining: 6m 22s
600:	learn: 0.6440200	test: 0.6383938	best: 0.6383938 (600)	total: 23.4s	remaining: 6m 6s
800:	learn: 0.6505291	test: 0.6431404	best: 0.6431404 (800)	total: 31.1s	remaining: 5m 56s
1000:	learn: 0.6560548	test: 0.6469170	best: 0.6469170 (1000)	total: 38.5s	remaining: 5m 45s
1200:	learn: 0.6610493	test: 0.6500372	best: 0.6500372 (1200)	total: 45.8s	remaining: 5m 35s
1400:	learn: 0.6655781	test: 0.6530196	best: 0.6530196 (1400)	total: 52.9s	remaining: 5m 24s
1600:	learn: 0.6692451	test: 0.6549461	best: 0.6549461 (1600)	total: 1m	remaining: 5m 19s
1800:	learn: 0.6728623	test: 0.6568739	best: 0.6568739 (1800)	total: 1m 8s	remaining: 5m 13s
2000:	learn: 0.6765179	test: 0.6589962	best: 0.6589962 (2000)	total: 1m 16s	remaining: 5m

0.596060992744249

In [17]:
#model = CatBoostClassifier(loss_function = 'Logloss', 
 #                          eval_metric='AUC', 
  #                       task_type="GPU",
   #                        use_best_model=True,
    #                       metric_period=1,
     #                      random_seed=6006)

In [18]:
#model.fit(train.iloc[:, :130], train.iloc[:, 130:], eval_set=(val.iloc[:, :130], val.iloc[:, 130:]))

In [19]:
drop_cols = ['date', 'weight']

In [20]:
import janestreet
env = janestreet.make_env() # initialize the environment
iter_test = env.iter_test() # an iterator which loops over the test set



In [21]:
for (test_df, sample_prediction_df) in iter_test:
    X_test = test_df.drop(drop_cols, axis=1)
    X_test = X_test.fillna(0)
    y_preds = m.predict(X_test)
    sample_prediction_df.action = y_preds
    env.predict(sample_prediction_df)